# 4 · PII screening of free text

Notes, complaints and call transcripts land in the lakehouse with names, health or financial details inside. Regex finds BSN and IBAN; it can't read *mijn buurvrouw mevrouw de Vries heeft diabetes*.

**The pattern:** regex with checksums settles structured identifiers → Jev answers one Noul per PII category for the rest → code routes each row: *masked*, *restricted* or *excluded* from gold.

In [ ]:
import json

import pandas as pd

from jevdemo import pii
from jevdemo.judge import MODEL, Judge, LiveUnavailable, NotRecorded, request_body

judge = Judge()  # JEV_MODE=live for live calls; default replays recordings
print(f"mode: {judge.mode} · model: {MODEL}")

## The notes
22 synthetic Dutch notes. The category columns are our hand labels, used only to score afterwards.

In [ ]:
notes = pii.load_notes()
pd.DataFrame(notes)

## Code step: regex with checksums
BSN must pass the 11-check and IBAN mod-97, so an order number doesn't count. A hit is a definite flag and the row skips Jev.

In [ ]:
hits = {n["id"]: pii.regex_hits(n["text"]) for n in notes}
pd.DataFrame(
    [
        {"id": i, "text": n["text"], "regex": sorted(hits[i])}
        for n in notes
        if (i := n["id"]) and hits[i]
    ]
)

## What Jev receives
One request per note with five Nouls, one per category. They're evaluated in parallel, so five questions cost about the same time as one.

In [ ]:
print(
    json.dumps(
        request_body(*pii.screen_request(notes[0]["text"])),
        indent=2,
        ensure_ascii=False,
    )
)

## Jev screens every note regex didn't settle

In [ ]:
rest = [n for n in notes if not hits[n["id"]]]
answers = judge.ask_many([pii.screen_request(n["text"]) for n in rest])
probs = {
    n["id"]: {c: a.response.nouls[c].noul for c in pii.CATEGORIES}
    for n, a in zip(rest, answers)
}
pd.DataFrame([{"id": i, **p} for i, p in probs.items()]).set_index("id").round(2)

## Routing
A category counts at `p ≥ 0.30`, low on purpose: missing PII costs more than a review. Health and Art. 9 data → **excluded** from gold. Other categories Jev finds → **restricted** table, because Jev says *whether*, not *where*, so code can't mask the span. Regex hits → **masked**, because code knows the span.

In [ ]:
routed = pd.DataFrame(
    [
        {
            "id": n["id"],
            "text": n["text"],
            "contains_pii": (r := pii.route(hits[n["id"]], probs.get(n["id"], {})))
            != "clear",
            "route": r,
        }
        for n in notes
    ]
)
print(routed["route"].value_counts().to_string())
routed

## Contextual PII: regex alone vs regex + Jev

In [ ]:
contextual = [n for n in notes if any(n[c] for c in pii.CATEGORIES)]
by_regex = sum(bool(hits[n["id"]]) for n in contextual)
by_jev = sum(routed.set_index("id").loc[n["id"], "contains_pii"] for n in contextual)
print(
    f"notes with contextual PII: {len(contextual)} · regex caught {by_regex} · regex + Jev caught {by_jev}"
)

## Per category, against our labels
Recall is the metric that matters here; precision is what the low threshold costs.

In [ ]:
rows = []
for c in pii.CATEGORIES:
    flagged = {i for i, p in probs.items() if p[c] >= pii.THRESHOLD}
    truth = {n["id"] for n in rest if n[c]}
    rows.append(
        {
            "category": c,
            "labelled": len(truth),
            "caught": len(flagged & truth),
            "extra flags": len(flagged - truth),
            "recall": len(flagged & truth) / len(truth) if truth else None,
        }
    )
pd.DataFrame(rows).set_index("category")

## Where Jev flagged more than we labelled
Worth reading out loud: some are arguably right (an unnamed *meneer* is still a private individual), and they show the cost of leaning towards recall.

In [ ]:
extra = [
    {
        "id": i,
        "category": c,
        "p": round(p[c], 2),
        "text": next(n["text"] for n in rest if n["id"] == i),
    }
    for i, p in probs.items()
    for c in pii.CATEGORIES
    if p[c] >= pii.THRESHOLD and not next(n for n in rest if n["id"] == i)[c]
]
pd.DataFrame(extra)

## Your turn
Type a note. The preset is recorded; a new note needs `JEV_MODE=live`.

In [ ]:
text = pii.LIVE_PRESET  # or type your own

regex = pii.regex_hits(text)
if regex:
    print(f"regex: {sorted(regex)} → {pii.route(regex, {})} (Jev not asked)")
else:
    try:
        answer = judge.ask(*pii.screen_request(text))
        p = {c: answer.response.nouls[c].noul for c in pii.CATEGORIES}
        print({c: round(v, 2) for c, v in p.items()})
        print(f"→ {pii.route(set(), p)}  ({answer.source}, {answer.latency_ms} ms)")
    except (NotRecorded, LiveUnavailable) as e:
        print(e)

## The gate before client data

This demo runs on synthetic notes only. Sending client text to an external API *to find PII* needs the data-residency question settled first: retention, hosting region and the processing agreement. Until then, this stays a synthetic demo.

Two limits to name:
- A regex hit skips Jev, so a note with a phone number *and* health details is only masked. On real data you'd still ask Jev about those rows.
- Jev says *whether* a note contains PII, not *where*. Masking the span is a later step.

More depth: the **jevex** demo (`../jevex`) compares regex and Jev station by station on customer-service email.